# TEMREN Görev 3: Görüntü Eşleme — Hiperparametre Optimizasyonu

Bu notebook, Temren takımının Teknofest 2026 Havacılıkta Yapay Zeka Görev 3 (Görüntü Eşleme) algoritmasını (**DINOv2 + LoFTR + CSRT**) örnek görüntüler üzerinde test etmek ve hiperparametrelerini optimize etmek için hazırlanmıştır.

**Çalışma Ortamı:** Menüden `Runtime (Çalışma Zamanı)` -> `Change runtime type (Çalışma zamanı türünü değiştir)` seçip donanım hızlandırıcısını **T4 GPU** olarak ayarlayın.

In [ ]:
# 1. GPU Kontrolü
!nvidia-smi

### 2. Repo Klonlama ve Bağımlılıkların Kurulumu

In [ ]:
import os

if not os.path.exists('Temren'):
    !git clone https://github.com/necmettinsnmz/Temren.git

%cd Temren
!git pull origin main

# Bağımlılıkları kur
!pip install -r requirements.txt

### 3. Test Görüntülerinin İndirilmesi
Farklı açı ve ölçek farkı içeren örnek sahne görselleri indirilir.

In [ ]:
import urllib.request
import os
import cv2
import matplotlib.pyplot as plt

os.makedirs("test_data", exist_ok=True)

ref_url = "https://raw.githubusercontent.com/cvg/LightGlue/main/assets/sacre_coeur1.jpg"
frame_url = "https://raw.githubusercontent.com/cvg/LightGlue/main/assets/sacre_coeur2.jpg"

urllib.request.urlretrieve(ref_url, "test_data/ref.jpg")
urllib.request.urlretrieve(frame_url, "test_data/frame.jpg")

print("Test verileri indirildi!")

### 4. DINOv2 + LoFTR Pipeline Testi

In [ ]:
import sys
sys.path.append('.')

from tasks.task3_matching import ImageMatcher
import time

# Matcher'ı başlat (DINOv2 ve LoFTR ağırlıkları indirilecektir)
matcher = ImageMatcher(device="cuda")

ref_img = cv2.imread("test_data/ref.jpg")
frame = cv2.imread("test_data/frame.jpg")

# Referans görüntüden örnek bir alt bölge kırpalım
h, w = ref_img.shape[:2]
ref_cropped = ref_img[h//3:2*h//3, w//3:2*w//3].copy()

start_time = time.time()
detections = matcher.run(frame, reference_objects=[{"id": 1, "image": ref_cropped}])
end_time = time.time()

output = frame.copy()
if detections:
    det = detections[0]
    x1, y1 = int(det["top_left_x"]), int(det["top_left_y"])
    x2, y2 = int(det["bottom_right_x"]), int(det["bottom_right_y"])
    cv2.rectangle(output, (x1, y1), (x2, y2), (0, 255, 0), 4)

    plt.figure(figsize=(15, 8))
    plt.subplot(1, 2, 1)
    plt.title("Referans Obje")
    plt.imshow(cv2.cvtColor(ref_cropped, cv2.COLOR_BGR2RGB))

    plt.subplot(1, 2, 2)
    plt.title(f"Tespit Süresi: {(end_time - start_time)*1000:.1f}ms")
    plt.imshow(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
    plt.show()
else:
    print("Eşleşme bulunamadı! config.py içindeki eşik değerlerini gözden geçirin.")

### 5. Hiperparametre Optimizasyonu

Modelin eşleşme performansı veya aday kapsamı yetersiz gelirse `config.py` dosyasındaki parametreleri güncelleyip 4. hücreyi tekrar çalıştırın:
1. `DINOV2_SIMILARITY_THRESHOLD`: Kaba arama benzerlik eşiği (varsayılan `0.20`; daha fazla aday için düşürün).
2. `LOFTR_CONFIDENCE_THRESHOLD`: İnlier oranı güven eşiği (varsayılan `0.20`).
3. `LOFTR_MIN_MATCHES`: Homografi doğrulaması için minimum eşleşme sayısı (varsayılan `6`).
4. `CANDIDATE_SCALES`: Aday bölge arama ölçekleri (varsayılan `[1.0, 1.5, 2.5]`).